# Pairs Trading Walkthrough

This notebook runs the complete educational workflow on synthetic data: chronological splitting, training-only OLS, trailing z-scores, lagged positions, transaction costs, out-of-sample metrics, and plots.

**This is a toy research example—not financial advice or a live-trading system.**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from pairs_trading.backtest import BacktestConfig, plot_results, run_backtest
from pairs_trading.data import generate_synthetic_prices, load_price_csv, train_test_split

## 1. Generate data and split time chronologically

`asset_a` is a geometric random walk. `asset_b` is built from `asset_a` plus an AR(1) residual, so the example begins with a deliberately favorable mean-reverting relationship. The first 60% is training data and the later 40% is an untouched test period.

In [ ]:
prices = generate_synthetic_prices(n_periods=1_000, seed=42)
training_prices, test_prices = train_test_split(prices, train_fraction=0.60)

print(f"Training: {training_prices.index.min().date()} to {training_prices.index.max().date()} ({len(training_prices)} rows)")
print(f"Test:     {test_prices.index.min().date()} to {test_prices.index.max().date()} ({len(test_prices)} rows)")
prices.head()

## 2. Fit on training data; simulate on test data

OLS estimates $B_t = \alpha + \beta A_t + \varepsilon_t$ only on training rows. The residual $S_t = B_t-(\hat\alpha+\hat\beta A_t)$ is normalized with a trailing 20-day z-score. Signals enter at ±2 and exit inside ±0.5. A close-based signal is lagged one period before it earns P&L.

In [ ]:
config = BacktestConfig(
    zscore_window=20,
    entry_z=2.0,
    exit_z=0.5,
    transaction_cost_bps=5.0,
)
result = run_backtest(training_prices, test_prices, config=config)

print(f"OLS intercept:   {result.model.intercept:.4f}")
print(f"OLS hedge ratio: {result.model.hedge_ratio:.4f}")

## 3. Inspect the required metrics

Returns are gross-exposure normalized, costs are proportional to position turnover, annualization assumes 252 periods per year, and the Sharpe ratio assumes a zero risk-free rate.

In [ ]:
metrics = pd.Series(result.metrics, name="value")
metrics

## 4. Plot prices, spread/z-score, position, and equity

Prices are normalized to 100 only for visual comparison. The equity curve includes the configured transaction costs.

In [ ]:
figure, axes = plot_results(
    prices,
    result,
    entry_z=config.entry_z,
    exit_z=config.exit_z,
)
plt.show()

## 5. Transaction-cost sensitivity

This comparison illustrates implementation drag. It is not a parameter-optimization exercise.

In [ ]:
rows = []
for cost_bps in (0.0, 5.0, 10.0, 25.0):
    scenario = run_backtest(
        training_prices,
        test_prices,
        config=BacktestConfig(transaction_cost_bps=cost_bps),
    )
    rows.append({"cost_bps": cost_bps, **scenario.metrics})

pd.DataFrame(rows).set_index("cost_bps")

## Optional, clearly separated: load a local historical CSV

The default workflow above uses no API and no credentials. If you have a lawful local CSV with a date and two price columns, adapt the disabled cell below. Historical data introduces timestamp, adjustment, delisting, and licensing questions that synthetic data avoids.

In [ ]:
if False:  # Change only after supplying your own local file.
    historical_prices = load_price_csv(
        "path/to/your/prices.csv",
        date_column="date",
        price_columns=("stock_x", "stock_y"),
    )
    historical_training, historical_test = train_test_split(
        historical_prices, train_fraction=0.60
    )

## Research warnings

- **Look-ahead bias:** fitting on test data, using future values in rolling statistics, or earning same-bar returns from a close-based signal leaks information. This workflow uses training-only OLS, trailing windows, and a one-period P&L lag.
- **Overfitting:** repeatedly tuning windows and thresholds can fit noise.
- **Data snooping:** after repeated inspection, a test set is no longer truly unseen.
- **Survivorship bias:** a universe containing only present-day survivors omits historical failures and delistings.
- **Cointegration breakdown:** business changes, regulation, mergers, shocks, or regime shifts can permanently change the relationship and make a fixed hedge ratio obsolete.

The synthetic construction is intentionally favorable. Real trading also faces slippage, borrow availability and fees, financing, market impact, taxes, latency, missing data, and execution risk.